In [ ]:
# ==========================================
# 1. PERSIAPAN RUANG & DOWNLOAD DATASET
# ==========================================
from google.colab import files
import os

# Upload kaggle.json
uploaded = files.upload()
for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))

# Pindahkan kaggle.json ke folder .kaggle agar terbaca oleh API
!mkdir -p ~/.kaggle/ && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Download dataset dari Kaggle
!kaggle datasets download -d gpiosenka/butterfly-images40-species

# Extract dataset
!unzip /content/butterfly-images40-species.zip

# Pindahkan ke Google Drive (Pastikan Drive sudah di-mount sebelumnya)
!cp -r butterflies '/content/drive/MyDrive/TUGAS BESAR SCL/'

# Ubah working directory ke folder di Drive
%cd /content/drive/MyDrive/TUGAS BESAR SCL/
!ls

# Import Library
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import EarlyStopping
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import Callback
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd

In [ ]:
# ==========================================
# 2. EKSPLORASI DAN PERSIAPAN DATASET
# ==========================================
train_dir = "butterflies/train"
valid_dir = "butterflies/valid"

# Dataset exploration
explor_data = []
labels = []
testing_path = 'butterflies/test'

for label in os.listdir(testing_path): # loop masing masing class
    path = testing_path+'/'+label
    img_paths = os.listdir(path)
    random.shuffle(img_paths)
    img_path = path+'/'+img_paths[0]
    img = plt.imread(img_path) # load image
    explor_data.append(img) # masukan citra kedalam list
    labels.append(label) # masukan encoded label kedalam list

explor_data = np.array(explor_data)
print("Shape explor data:", explor_data.shape)

# Menampilkan sampel gambar
plt.figure(figsize=(20,20))
for i in range(50):
    plt.subplot(7,8,i+1)
    plt.title(labels[i])
    plt.axis('off')
    plt.imshow(explor_data[i])
plt.show()

In [ ]:
# ==========================================
# 3. AUGMENTASI DATA
# ==========================================
# Melakukan augmentasi pada dataset
datagen = ImageDataGenerator(
    rescale=1./255, # normalisasi ke float 0-1
    rotation_range=20, # random rotation range max 20 derajat
    zoom_range=0.2, # zoom range 0.2 dari ukuran asli
    shear_range=0.2, # shear
    width_shift_range=0,
    height_shift_range=0,
    vertical_flip=False,
    fill_mode='nearest'
)

# Load dataset secara bertahap
train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

validation_generator = datagen.flow_from_directory(
    valid_dir,
    target_size=(224,224),
    batch_size=32,
    shuffle=False
)

print("Class indices:", train_generator.class_indices)

In [ ]:
# ==========================================
# 4. PERSIAPAN CALLBACKS
# ==========================================
# Setup folder penyimpanan model
modelFolder = 'model[0]/' # Sesuaikan dengan model yang sedang di-train
os.makedirs(modelFolder, exist_ok=True)

checkpointLoss = ModelCheckpoint(f"{modelFolder}bestLoss.hdf5",
    monitor='loss',
    verbose=1,
    save_best_only=True,
    mode='auto'
)

checkpointValLoss = ModelCheckpoint(f"{modelFolder}bestValLoss.hdf5",
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    mode='auto'
)

earlyStopVal = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)
earlyStop = EarlyStopping(monitor='loss', mode='min', verbose=1, patience=10)

# Custom callback untuk akurasi > 95%
class myCallback(Callback):
    def on_epoch_end(self, epoch, logs={}):
        if(logs.get('accuracy') > 0.95 and logs.get('loss') < 0.001):
            print("\nAkurasi telah mencapai > 95%! dan loss < 0.001")
            self.model.stop_training = True

callbacks = myCallback()

In [ ]:
# ==========================================
# 5. MODEL 1: SEQUENTIAL
# ==========================================
print("\n--- MEMULAI TRAINING MODEL SEQUENTIAL ---")
model = models.Sequential([
    # EXTRACTION LAYER
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(224, 224, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(256, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Dropout(0.2),

    # CLASSIFICATION LAYER
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(50, activation='sigmoid') # Note: Sesuai laporan
])

model.summary()

model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=60,
    callbacks=[callbacks, checkpointValLoss, earlyStopVal]
)

model.save('lastModel')

df = pd.DataFrame()
df["accuracy"] = history.history["accuracy"]
df["val_accuracy"] = history.history['val_accuracy']
df["loss"] = history.history['loss']
df["val_loss"] = history.history["val_loss"]
df.to_csv(f"{modelFolder}history.csv")

In [ ]:
# ==========================================
# 6. MODEL 2: INCEPTION RESNET V2
# ==========================================
print("\n--- MEMULAI TRAINING MODEL INCEPTION ---")
modelFolder = 'model[1]/' # Ubah direktori untuk model kedua
os.makedirs(modelFolder, exist_ok=True)

# Update callbacks path
checkpointValLoss = ModelCheckpoint(f"{modelFolder}bestValLoss.hdf5", monitor='val_loss', verbose=1, save_best_only=True, mode='auto')

pretrained = tf.keras.applications.InceptionResNetV2(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3),
    classifier_activation='softmax'
)

modelInception = tf.keras.Sequential([
    pretrained,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(50, activation='softmax')
])

modelInception.summary()

modelInception.compile(loss='categorical_crossentropy',
                       optimizer='rmsprop',
                       metrics=['accuracy'])

history2 = modelInception.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=60,
    callbacks=[callbacks, checkpointValLoss, earlyStopVal]
)

modelInception.save('inception')

df_inc = pd.DataFrame()
df_inc["accuracy"] = history2.history["accuracy"]
df_inc["val_accuracy"] = history2.history['val_accuracy']
df_inc["loss"] = history2.history['loss']
df_inc["val_loss"] = history2.history["val_loss"]
df_inc.to_csv(f"{modelFolder}history.csv")